In [1]:
!pip install datasets[audio] librosa transformers

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/Speaker-Recognition-using-ResNet-Embeddings

/content/drive/MyDrive/Speaker-Recognition-using-ResNet-Embeddings


# Data Preprocessing
loading data from local and Preprocess

In [4]:
from datasets import load_from_disk, concatenate_datasets, Audio

import glob

# loading data

drive_path = "/content/drive/MyDrive/Speaker-Recognition-using-ResNet-Embeddings/speaker_chunks/chunk_*"
chunk_paths = sorted(glob.glob(drive_path))

chunks = []
for path in chunk_paths:
    ds = load_from_disk(path)
    ds = ds.cast_column("audio_path", Audio(sampling_rate=16000))
    chunks.append(ds)

full_dataset = concatenate_datasets(chunks)

In [13]:
from sklearn.model_selection import StratifiedKFold, train_test_split
import numpy as np

# fixed held-out test set
split = full_dataset.train_test_split(test_size=0.1, seed=42)
trainval_dataset = split["train"]   # for CV
test_dataset = split["test"]  # fixed

# get the labels for stratification
labels = np.array(trainval_dataset["label"])

# 9-way stratified split, take 3 folds
skf = StratifiedKFold(n_splits=9, shuffle=True, random_state=42)
fold_indices = list(skf.split(np.zeros(len(labels)), labels))

n_folds = 3
folds = [] #collect the 3 folds
for i in range(n_folds):
    train_idx, dev_idx = fold_indices[i]
    train_dataset = trainval_dataset.select(train_idx.tolist())
    dev_dataset = trainval_dataset.select(dev_idx.tolist())
    folds.append((train_dataset, dev_dataset)) #collect the 3 folds
    print(f"Fold {i}: train={len(train_dataset)}, dev={len(dev_dataset)}, test={len(test_dataset)}")

22500 2500
Fold 0: train=20000, dev=2500, test=2500
Fold 1: train=20000, dev=2500, test=2500
Fold 2: train=20000, dev=2500, test=2500


In [16]:
from datasets import disable_caching

# disable caching so no addition file are saved in drives

disable_caching()

Prepocess data by using Dataset_Builder script

In [22]:
import sys
from google.colab import drive

# Add the specific folder containing dataset.py to Python's search path
sys.path.append('/content/drive/MyDrive/Speaker-Recognition-using-ResNet-Embeddings')

from dataset import Dataset_Builder

# try the training set first

config = {"shortest_duration": 4.0, "timesteps": 400} # (optional)

dataset_train_folds = []

#loop for 3 folds
for fold in folds:
  train_builder = Dataset_Builder(fold[0], **config)
  train_builder.filter()
  train_builder.preprocess()
  dataset_train_folds.append(train_builder.get_dataset())

Duration of cropped log-mel spectograms (input):  4.0  seconds


Filter:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/19232 [00:00<?, ? examples/s]

Duration of cropped log-mel spectograms (input):  4.0  seconds


Filter:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/19241 [00:00<?, ? examples/s]

Duration of cropped log-mel spectograms (input):  4.0  seconds


Filter:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/19239 [00:00<?, ? examples/s]

In [23]:
# Also do it for dev, and test
dataset_dev_folds = []

#loop for 3 folds
for fold in folds:
  dev_builder = Dataset_Builder(fold[1], **config)
  dev_builder.filter()
  dev_builder.preprocess()
  dataset_dev_folds.append(dev_builder.get_dataset())

test_builder = Dataset_Builder(test_dataset, **config)
test_builder.filter()
test_builder.preprocess()

Duration of cropped log-mel spectograms (input):  4.0  seconds


Filter:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2407 [00:00<?, ? examples/s]

Duration of cropped log-mel spectograms (input):  4.0  seconds


Filter:   0%|          | 0/2500 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Duration of cropped log-mel spectograms (input):  4.0  seconds


Filter:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2401 [00:00<?, ? examples/s]

In [25]:
# get the proprocess data

for i in range(len(folds)):
  print(f"Fold {i}: train={len(dataset_train_folds[i])}, dev={len(dataset_dev_folds[i])}")

test_dataset = test_builder.get_dataset()
print(f"test={len(test_dataset)}")

Fold 0: train=19232, dev=2407
Fold 1: train=19241, dev=2398
Fold 2: train=19239, dev=2400
test=2401


## Create DataLoaders

Create a custom collate function to convert the data into tensors. Then, build the train, dev, and test data loaders and check that the batch dimensions match the expected input format.

In [26]:
import torch
from torch.utils.data import DataLoader

NUM_CLASSES = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def collate_fn(batch):
    log_mels = torch.stack([b["log_mel"] for b in batch])
    labels = torch.tensor([b["label"] for b in batch])
    return (log_mels, labels)

dataloader_train_folds = []
dataloader_dev_folds = []

#loop for folds
for train_dataset, dev_dataset in zip(dataset_train_folds, dataset_dev_folds):
    dataloader_train = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=False,  # Already shuffled before saving
        collate_fn=collate_fn,
        pin_memory=True
    )
    dataloader_dev = DataLoader(
        dev_dataset,
        batch_size=32,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True
    )

    dataloader_train_folds.append(dataloader_train)
    dataloader_dev_folds.append(dataloader_dev)

### Seed Everything

In [27]:
import random
import numpy as np
import torch

SEED = 42

def seed_everything(seed=42):
    # Python
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch CPU
    torch.manual_seed(seed)

    # PyTorch GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    # CuDNN
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

# Training

### Load Pretrained Model, Optimizer, Scheduler, Loss

In [28]:
from torchvision.models import resnet18
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from torch.utils.tensorboard import SummaryWriter

# lr: 5e-5
LR = 5e-5

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## Training loop
I've managed to run it for the first fold, so the code is for the remaining two.

In [ ]:
from tqdm import tqdm
import os

NUM_EPOCHS = 6
NUM_FOLDS = len(dataloader_train_folds)

for fold_idx, (dataloader_train, dataloader_dev) in enumerate(zip(dataloader_train_folds[1:], dataloader_dev_folds[1:])):
    print(f"\n=========== FOLD {fold_idx} ===========")

    # --- Re-initialize everything fresh for this fold ---
    model = resnet18(weights="DEFAULT")
    model.fc = torch.nn.Linear(in_features=512, out_features=NUM_CLASSES, bias=True)
    model.to(DEVICE)
    optimizer = AdamW(params=model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
    ce_loss = CrossEntropyLoss()

    writer = SummaryWriter(log_dir=f"runs/fold_{fold_idx}")  # separate TB log per fold

    model = model.to(DEVICE)
    best_val_acc = 0.0
    best_ckpt_path = f"best_speaker_resnet_fold{fold_idx}.pth"

    for epoch in range(1, NUM_EPOCHS+1):
        print(f"--- Epoch: {epoch} ---")

        model.train()
        for indx, (log_mels, labels) in tqdm(enumerate(dataloader_train), desc="Epoch Progress", total=len(dataloader_train)):
            log_mels = log_mels.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            preds = model(log_mels)
            loss = ce_loss(preds, labels)
            loss.backward()
            optimizer.step()

            global_step = (epoch-1)*len(dataloader_train) + indx
            writer.add_scalar("Loss/Train", loss.item(), global_step)
        scheduler.step()

        # ==================== VALIDATION PHASE ====================
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for log_mels, labels in tqdm(dataloader_dev, desc="Validation", total=len(dataloader_dev)):
                log_mels = log_mels.to(DEVICE)
                labels = labels.to(DEVICE)

                preds = model(log_mels)
                loss = ce_loss(preds, labels)

                val_loss += loss.item()
                val_correct += torch.sum(torch.argmax(preds, dim=1) == labels).item()
                val_total += len(labels)

        avg_val_loss = val_loss / len(dataloader_dev)
        val_acc = val_correct / val_total

        print(f"Fold {fold_idx} | Epoch {epoch} finished | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

        writer.add_scalar("Loss/Val", avg_val_loss, epoch)
        writer.add_scalar("Accuracy/Val", val_acc, epoch)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_ckpt_path)
            print(f"🌟 New best model saved for fold {fold_idx}! (Val Acc: {val_acc:.4f})")

    writer.close()
    print(f"Fold {fold_idx} finished | Best Val Acc: {best_val_acc:.4f}")


=========== FOLD 0 ===========
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 134MB/s]


--- Epoch: 1 ---


Validation: 100%|██████████| 76/76 [04:03<00:00,  3.21s/it]


Fold 0 | Epoch 1 finished | Val Loss: 1.0904 | Val Acc: 0.7640
🌟 New best model saved for fold 0! (Val Acc: 0.7640)
--- Epoch: 2 ---


...

Validation: 100%|██████████| 76/76 [04:00<00:00,  3.17s/it]


Fold 0 | Epoch 6 finished | Val Loss: 0.2573 | Val Acc: 0.9356
🌟 New best model saved for fold 0! (Val Acc: 0.9356)
Fold 0 finished | Best Val Acc: 0.9356

=========== FOLD 1 ===========
--- Epoch: 1 ---


Validation: 100%|██████████| 75/75 [03:59<00:00,  3.19s/it]


Fold 1 | Epoch 1 finished | Val Loss: 1.0890 | Val Acc: 0.7598
🌟 New best model saved for fold 1! (Val Acc: 0.7598)
--- Epoch: 2 ---


Epoch Progress:  46%|████▌     | 274/602 [15:05<17:09,  3.14s/it]

In [ ]:
# from tqdm import tqdm

# model.eval()

# correct  = 0
# total = 0
# for indx, (log_mels, labels) in tqdm(enumerate(dataloader_test), total=len(dataloader_test)):
#     log_mels = log_mels.to(DEVICE)
#     labels = labels.to(DEVICE)
#     pred = model(log_mels)

#     correct += torch.sum(torch.argmax(pred, dim=1) == labels).item()
#     total += len(labels)

# accuracy = correct / total
# print(f"Accuracy = {accuracy}")

100%|██████████| 10/10 [00:28<00:00,  2.89s/it]

Accuracy = 0.25597269624573377
